# 04 - Experimental Matrix Audit & Monitoring Dashboard (Dynamic DB-Driven)

**Scientific & Operational Objectives:**
1. **Dynamic Database Discovery**: Directly queries  and  to detect dimensions, noise regimes, problem IDs, and model architectures without hardcoded lists.
2. **Two-Tier Pipeline Audit**:
   - **Tier 1 (Evolutionary Phase - SQLite)**: Audits LLaMEA evolutionary synthesis runs, iteration counts, convergence frequencies, and database integrity.
   - **Tier 2 (Evaluation Phase - IOH Logs)**: Audits post-evolution =10$ benchmark evaluations for LLM champions and classical baselines (, , ).
3. **Sample Imbalance & Gap Detection**: Quantifies run distribution imbalances and flags unexecuted experimental cells.
4. **Automated Report & Visual Export**:
   - Generates 
   - Renders 
5. **Actionable Task Dispatcher**: Emits ready-to-run commands for any missing experimental condition.


In [37]:
# ── 1. Environment Setup & Configuration ──────────────────────────────────
import os, sys, re, json, sqlite3
from datetime import datetime
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.kaleido.scope.default_scale = 3.0
pio.kaleido.scope.default_format = "png"

CWD = Path(".").resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.config import DATA_DIR, RESULTS_DIR

DB_PATH      = DATA_DIR / "db.sqlite3"
IOH_LOGS_DIR = DATA_DIR / "ioh_logs"
FIGURES_DIR  = RESULTS_DIR / "figures"
REPORTS_DIR  = RESULTS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

BBOB_NAMES_MAP = {
    1: "Sphere (f1)", 8: "Rosenbrock (f8)", 11: "Discus (f11)",
    15: "Rastrigin (f15)", 21: "Gallagher (f21)"
}
BBOB_CLASSES_MAP = {
    1: "Separable", 8: "Low Conditioning", 11: "High Conditioning",
    15: "Multi-Modal (Global)", 21: "Multi-Modal (Weak)"
}

print("✅ Dynamic Audit Environment initialized.")


✅ Dynamic Audit Environment initialized.


/var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/ipykernel_50076/4135273233.py:13: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_scale is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_scale instead.

  pio.kaleido.scope.default_scale = 3.0
/var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/ipykernel_50076/4135273233.py:14: DeprecationWarning: 
Use of plotly.io.kaleido.scope.default_format is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.default_format instead.

  pio.kaleido.scope.default_format = "png"


In [38]:
# ── 2. Dynamic Database & File System Discovery ───────────────────────────
from types import SimpleNamespace

def map_db_solver_name(llm_name: str, strat: str) -> str:
    l, s = str(llm_name).lower(), str(strat).lower()
    if "7b" in l: return "LLaMEA-7B / baseline"
    if "14b" in l: return f"LLaMEA-14B / {s}"
    if "70b" in l: return f"LLaMEA-70B / {s}"
    return f"{llm_name} / {s}"

def resolve_ioh_solver_name(parent_name: str) -> str:
    p = parent_name.lower()
    if "cmaes" in p or "cma_es" in p: return "CMA-ES"
    if "pso" in p: return "PSO"
    if p == "de" or p.startswith(("de_", "de-")) or "_de_" in p: return "DE"
    if "7b" in p: return "LLaMEA-7B / baseline"
    if "thinking" in p: return "LLaMEA-14B / thinking"
    if "vectorization" in p: return "LLaMEA-14B / vectorization"
    if "guided" in p: return "LLaMEA-14B / guided"
    if "14b" in p or "baseline" in p or "llamea" in p: return "LLaMEA-14B / baseline"
    return parent_name

def load_audit_data(db_path: Path, ioh_dir: Path) -> SimpleNamespace:
    df_exp, df_iter = pd.DataFrame(), pd.DataFrame()
    ioh_counts = defaultdict(lambda: defaultdict(int))
    dims, noise_levels, problem_ids, llm_solvers = [], [], [], []

    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            df_exp = pd.read_sql_query("SELECT * FROM experiments;", conn)
            df_iter = pd.read_sql_query(
                "SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, "
                "i.raw_fitness, i.final_error, i.timed_out, i.converged, i.runtime_seconds, "
                "e.problem_id, e.dim, e.mode, e.llm_name, e.prompt_strategy, e.noise_std "
                "FROM iterations i JOIN experiments e ON i.experiment_id = e.id;", conn
            )
        if not df_exp.empty:
            df_exp["solver_name"] = df_exp.apply(
                lambda r: map_db_solver_name(r["llm_name"], r["prompt_strategy"]), axis=1
            )
            dims = sorted(df_exp["dim"].unique().tolist())
            noise_levels = sorted(df_exp["noise_std"].unique().tolist())
            problem_ids = sorted(df_exp["problem_id"].unique().tolist())
            llm_solvers = sorted(df_exp["solver_name"].unique().tolist())

    if ioh_dir.exists():
        for json_path in ioh_dir.glob("**/*.json"):
            try:
                with open(json_path) as f: meta = json.load(f)
            except Exception: continue
            path_str = str(json_path.relative_to(ioh_dir))
            dim_m = re.search(r"(\d+)D", path_str); dim = int(dim_m.group(1)) if dim_m else None
            noise_m = re.search(r"std_([\d\.]+)", path_str); noise_std = float(noise_m.group(1)) if noise_m else 0.0
            p_id = meta.get("function_id")
            if p_id is None:
                p_m = re.search(r"f(\d+)", path_str); p_id = int(p_m.group(1)) if p_m else None
            solver = resolve_ioh_solver_name(json_path.parent.name)

            if dim and dim not in dims: dims.append(dim)
            if noise_std not in noise_levels: noise_levels.append(noise_std)
            if p_id and p_id not in problem_ids: problem_ids.append(p_id)

            for sc in meta.get("scenarios", []):
                d = dim or sc.get("dimension")
                runs = sc.get("runs", [])
                ioh_counts[(d, noise_std, p_id)][solver] += len(runs) if runs else 1

    dims = sorted(set(dims))
    noise_levels = sorted(set(noise_levels))
    problem_ids = sorted(set(problem_ids))
    all_ioh = {s for cond in ioh_counts.values() for s in cond}
    classical = sorted([s for s in all_ioh if not any(k in s for k in ["LLaMEA", "dummy"])])
    llm_14b = sorted([s for s in llm_solvers if "14B" in s])
    llm_7b = sorted([s for s in llm_solvers if "7B" in s])
    llm_other = sorted([s for s in llm_solvers if s not in llm_14b and s not in llm_7b])

    return SimpleNamespace(
        db_path=db_path, ioh_dir=ioh_dir, df_exp=df_exp, df_iter=df_iter,
        ioh_counts=ioh_counts, dims=dims, noise_levels=noise_levels,
        problem_ids=problem_ids, llm_solvers=llm_solvers,
        classical_solvers=classical,
        all_solvers=llm_14b + llm_7b + llm_other + classical
    )

loader = load_audit_data(DB_PATH, IOH_LOGS_DIR)
print("🔍 Dynamic DB Discovery Summary:")
print(f"   • Dimensions ({len(loader.dims)}): {loader.dims}")
print(f"   • Noise Levels ({len(loader.noise_levels)}): {loader.noise_levels}")
print(f"   • Problem IDs ({len(loader.problem_ids)}): {loader.problem_ids}")
print(f"   • Discovered Solvers ({len(loader.all_solvers)}): {loader.all_solvers}")


🔍 Dynamic DB Discovery Summary:
   • Dimensions (3): [2, 3, 5]
   • Noise Levels (2): [0.0, 0.05]
   • Problem IDs (5): [1, 8, 11, 15, 21]
   • Discovered Solvers (8): ['LLaMEA-14B / baseline', 'LLaMEA-14B / guided', 'LLaMEA-14B / thinking', 'LLaMEA-14B / vectorization', 'LLaMEA-7B / baseline', 'CMA-ES', 'DE', 'PSO']


In [39]:
# ── 3. Compile Multi-Tier Experimental Audit DataFrames ───────────────────
eval_records = []
for d in loader.dims:
    for n in loader.noise_levels:
        for p in loader.problem_ids:
            for s in loader.all_solvers:
                cnt = loader.ioh_counts[(d, n, p)][s]
                eval_records.append({
                    "Dimension": f"{d}D",
                    "Noise": f"σ={n}",
                    "Problem_ID": p,
                    "Problem_Name": BBOB_NAMES_MAP.get(p, f"f{p}"),
                    "Hardness_Class": BBOB_CLASSES_MAP.get(p, "Unknown"),
                    "Solver": s,
                    "Evaluated_Runs": cnt,
                    "Status": "Complete" if cnt >= 10 else ("Partial" if cnt > 0 else "Missing")
                })

df_eval_audit = pd.DataFrame(eval_records)
total_cells = len(df_eval_audit)
completed_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] > 0])
missing_cells = len(df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0])
completion_pct = (completed_cells / total_cells) * 100.0 if total_cells > 0 else 0.0

# SQLite Evolutionary Phase Audit
if not loader.df_exp.empty:
    db_summary = loader.df_exp.groupby(["dim", "noise_std", "problem_id", "solver_name"]).size().reset_index(name="experiment_count")
    print(f"🧬 SQLite Evolutionary Experiments Logged: {len(loader.df_exp)} across {len(loader.df_iter)} iterations.")

print(f"🎯 Benchmark Evaluation Matrix (IOH Logs): {completed_cells}/{total_cells} cells ({completion_pct:.1f}% complete).")


🧬 SQLite Evolutionary Experiments Logged: 156 across 1542 iterations.
🎯 Benchmark Evaluation Matrix (IOH Logs): 202/240 cells (84.2% complete).


In [40]:
# ── 4. Dynamic Markdown Audit Report Generator ───────────────────────────
missing_cells_df = df_eval_audit[df_eval_audit['Evaluated_Runs'] == 0]
partial_cells_df = df_eval_audit[(df_eval_audit['Evaluated_Runs'] > 0) & (df_eval_audit['Evaluated_Runs'] < 10)]
target_cells_df  = df_eval_audit[df_eval_audit['Evaluated_Runs'] == 10]
over_cells_df    = df_eval_audit[df_eval_audit['Evaluated_Runs'] > 10]

n_total = len(df_eval_audit)
n_miss = len(missing_cells_df)
n_part = len(partial_cells_df)
n_target = len(target_cells_df)
n_over = len(over_cells_df)

report_lines = []
report_lines.append('# 📋 Experimental Matrix Coverage & Sample Imbalance Audit Report\n')
report_lines.append(f'**Generated:** `notebooks/04_experimental_audit.ipynb` | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
report_lines.append('### 📊 High-Level Status Breakdown')
report_lines.append(f'- **Total Experimental Cells Planned:** `{n_total}`')
report_lines.append(f'- 🔴 **Missing Conditions (0 Runs):** `{n_miss}` ({n_miss/n_total*100:.1f}%)')
report_lines.append(f'- 🟡 **Partial / Interrupted Runs (<10 Runs):** `{n_part}` ({n_part/n_total*100:.1f}%)')
report_lines.append(f'- 🟢 **Exact Target Met (N = 10 Runs):** `{n_target}` ({n_target/n_total*100:.1f}%)')
report_lines.append(f'- 🔵 **Over-Sampled / Imbalanced (N > 10 Runs):** `{n_over}` ({n_over/n_total*100:.1f}%)\n')
report_lines.append('---\n')

report_lines.append('## 1. Completion Rate by Dimension\n')
report_lines.append('| Dimension | Total Cells | Fully Completed (N ≥ 10) | Partial (<10) | Missing (0) | Completion Rate |')
report_lines.append('|---|---|---|---|---|---|')
for d in loader.dims:
    sub = df_eval_audit[df_eval_audit['Dimension'] == f'{d}D']
    tot = len(sub)
    full = len(sub[sub['Evaluated_Runs'] >= 10])
    part = len(sub[(sub['Evaluated_Runs'] > 0) & (sub['Evaluated_Runs'] < 10)])
    miss = len(sub[sub['Evaluated_Runs'] == 0])
    rate = ((full + part) / tot) * 100.0 if tot > 0 else 0.0
    report_lines.append(f'| **{d}D** | {tot} | {full} | {part} | {miss} | {rate:.1f}% |')
report_lines.append('\n---\n')

report_lines.append('## 2. Sample Size Imbalance by Solver\n')
report_lines.append('| Solver | Category | Evaluated Cells | Missing Cells | Mean Runs (N) | Min Runs | Max Runs |')
report_lines.append('|---|---|---|---|---|---|---|')
for s in loader.all_solvers:
    sub = df_eval_audit[df_eval_audit['Solver'] == s]
    tot = len(sub)
    comp = len(sub[sub['Evaluated_Runs'] > 0])
    miss = len(sub[sub['Evaluated_Runs'] == 0])
    runs_s = sub[sub['Evaluated_Runs'] > 0]['Evaluated_Runs']
    mean_n = runs_s.mean() if comp > 0 else 0.0
    min_n = runs_s.min() if comp > 0 else 0
    max_n = runs_s.max() if comp > 0 else 0
    arch = 'LLaMEA-14B' if '14B' in s else ('LLaMEA-7B' if '7B' in s else 'Classical Baseline')
    report_lines.append(f'| **{s}** | {arch} | {comp}/{tot} | {miss} | **{mean_n:.1f}** | {min_n} | {max_n} |')
report_lines.append('\n---\n')

report_lines.append('## 3. Actionable Checklist: Missing Experiments (0 Runs)\n')
if missing_cells_df.empty:
    report_lines.append('🎉 **No missing conditions! Full coverage achieved.**\n')
else:
    report_lines.append('| # | Dimension | Noise Regime | Problem Name | Problem Class | Solver | Action Required |')
    report_lines.append('|---|---|---|---|---|---|---|')
    for idx, (_, row) in enumerate(missing_cells_df.iterrows(), 1):
        report_lines.append(f'| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | {row["Hardness_Class"]} | **{row["Solver"]}** | 🔴 **Execute N=10 Runs** |')
report_lines.append('\n---\n')

report_lines.append('## 4. Actionable Checklist: Partial / Interrupted Experiments (< 10 Runs)\n')
if partial_cells_df.empty:
    report_lines.append('🎉 **No partial runs detected!**\n')
else:
    report_lines.append('| # | Dimension | Noise Regime | Problem Name | Solver | Completed Runs | Remaining to Target (10 - N) |')
    report_lines.append('|---|---|---|---|---|---|---|')
    for idx, (_, row) in enumerate(partial_cells_df.iterrows(), 1):
        rem = 10 - int(row['Evaluated_Runs'])
        report_lines.append(f'| {idx} | {row["Dimension"]} | {row["Noise"]} | {row["Problem_Name"]} | **{row["Solver"]}** | ⚠️ {row["Evaluated_Runs"]}/10 | 🟡 **Run +{rem} more** |')

report_path = REPORTS_DIR / 'experimental_coverage_audit.md'
with open(report_path, 'w') as f:
    f.write('\n'.join(report_lines))
print(f'✅ Dynamic Audit Report written to: {report_path}')


✅ Dynamic Audit Report written to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/experimental_coverage_audit.md


In [41]:
# ── 5. Render High-Contrast Faceted Experimental Status & Imbalance Matrix ─
n_rows = len(loader.dims)
n_cols = len(loader.noise_levels)

def get_status_code(cnt):
    if cnt == 0: return 0.0      # Missing (Red)
    elif cnt < 10: return 1.0    # Partial (Amber)
    elif cnt == 10: return 2.0   # Target Met (Green)
    else: return 3.0             # Over-Sampled / Imbalance (Blue)

def get_status_text(cnt):
    if cnt == 0: return '❌ 0'
    elif cnt < 10: return f'⚠️ {cnt}/10'
    elif cnt == 10: return '✅ 10'
    else: return f'🔷 {cnt}'

colorscale = [
    [0.00, '#FFCDD2'], [0.24, '#FFCDD2'], # Missing (Soft Red)
    [0.26, '#FFE082'], [0.49, '#FFE082'], # Partial (Soft Amber)
    [0.51, '#C8E6C9'], [0.74, '#C8E6C9'], # Target Met (Soft Green)
    [0.76, '#BBDEFB'], [1.00, '#BBDEFB']  # Over-Sampled / Imbalance (Soft Blue)
]

subplot_titles = []
coords_map = {}
for r_idx, d in enumerate(loader.dims, 1):
    for c_idx, n in enumerate(loader.noise_levels, 1):
        label = 'Clean (σ=0.0)' if n == 0.0 else f'Noisy (σ={n})'
        total = len(loader.all_solvers) * len(loader.problem_ids)
        done = sum(1 for s in loader.all_solvers for p in loader.problem_ids if loader.ioh_counts[(d, n, p)][s] > 0)
        pct = (done / total) * 100 if total > 0 else 0
        subplot_titles.append(f'<b>{d}D — {label}</b>  <span style="font-size:11px; color:#555;">({done}/{total} done · {pct:.0f}%)</span>')
        coords_map[(d, n)] = (r_idx, c_idx)

fig_cov = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.10,
    vertical_spacing=0.10
)

prob_labels = [BBOB_NAMES_MAP.get(p, f'f{p}') for p in loader.problem_ids]

for d in loader.dims:
    for n in loader.noise_levels:
        r, c = coords_map[(d, n)]
        z_vals, text_vals = [], []
        for s in loader.all_solvers:
            row_z, row_t = [], []
            for p in loader.problem_ids:
                cnt = loader.ioh_counts[(d, n, p)][s]
                row_z.append(get_status_code(cnt))
                row_t.append(get_status_text(cnt))
            z_vals.append(row_z)
            text_vals.append(row_t)
            
        fig_cov.add_trace(
            go.Heatmap(
                z=z_vals,
                x=prob_labels,
                y=loader.all_solvers,
                text=text_vals,
                texttemplate='<b>%{text}</b>',
                textfont=dict(size=11, family='Inter, Helvetica, Arial, sans-serif', color='#1E293B'),
                colorscale=colorscale,
                zmin=0.0, zmax=3.0,
                showscale=False,
                xgap=3, ygap=3
            ),
            row=r, col=c
        )
        if c == 1:
            fig_cov.update_yaxes(autorange='reversed', row=r, col=c, tickfont=dict(size=11, family='Inter, sans-serif'))
        else:
            fig_cov.update_yaxes(showticklabels=False, autorange='reversed', row=r, col=c)
        fig_cov.update_xaxes(tickangle=-20, row=r, col=c, tickfont=dict(size=10, family='Inter, sans-serif'))

fig_cov.update_layout(
    template='plotly_white',
    title=dict(
        text='<b>Experimental Matrix Coverage & Sample Imbalance Dashboard</b><br>' +
             '<sup><b>Legend:</b>  ' +
             '<span style="color:#D32F2F;">■</span> <b>❌ 0 (Missing)</b>   &nbsp;|&nbsp;   ' +
             '<span style="color:#F57C00;">■</span> <b>⚠️ &lt;10 (Partial)</b>   &nbsp;|&nbsp;   ' +
             '<span style="color:#2E7D32;">■</span> <b>✅ 10 (Target Met)</b>   &nbsp;|&nbsp;   ' +
             '<span style="color:#1976D2;">■</span> <b>🔷 &gt;10 (Over-Sampled / Imbalance)</b></sup>',
        x=0.02, y=0.98,
        font=dict(size=13, color='#0F172A', family='Inter, Helvetica, Arial, sans-serif')
    ),
    width=1240, height=320 * n_rows + 40,
    margin=dict(l=190, r=40, t=110, b=50),
    font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#0F172A')
)

out_p = FIGURES_DIR / 'experimental_coverage_matrix.png'
fig_cov.write_image(str(out_p), scale=3)
print(f'✅ High-legibility status matrix exported to {out_p}')


✅ High-legibility status matrix exported to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/experimental_coverage_matrix.png


In [42]:
# ── 6. Missing Experiment Action Plan & CLI Dispatcher ───────────────────
missing_df = df_eval_audit[df_eval_audit["Evaluated_Runs"] == 0]
if not missing_df.empty:
    print(f"⚠️ {len(missing_df)} Experimental Conditions Require Execution:")
    for _, r in missing_df.iterrows():
        print(f"  • Dimension {r['Dimension']} | {r['Noise']} | {r['Problem_Name']} | Solver: {r['Solver']}")
else:
    print("🎉 All experimental matrix conditions are completely evaluated!")


⚠️ 38 Experimental Conditions Require Execution:
  • Dimension 2D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Rastrigin (f15) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.0 | Gallagher (f21) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.05 | Sphere (f1) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.05 | Rosenbrock (f8) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.05 | Discus (f11) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.05 | Rastrigin (f15) | Solver: LLaMEA-7B / baseline
  • Dimension 2D | σ=0.05 | Gallagher (f21) | Solver: LLaMEA-7B / baseline
  • Dimension 3D | σ=0.0 | Rosenbrock (f8) | Solver: LLaMEA-7B / baseline
  • Dimension 3D | σ=0.0 | Discus (f11) | Solver: LLaMEA-7B / baseline
  • Dimension 3D | σ=0.0 | Rastrigin (f15) | Solver: LLaMEA-7B / baseline
  • Dimension 3D | σ=0.0 | Gallagher (f21) | Solver: LLaMEA-7B / baseli